### Introduction to Preprocessing

Preprocessing refers to TRANSFORMING or MODIFYING variables **BEFORE** running or executing an analysis.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import seaborn as sns

## Read data

Use the Penguins data.

In [ ]:
penguins = sns.load_dataset('penguins')

In [ ]:
penguins.info()

In [ ]:
sns.pairplot(data = penguins)

plt.show()

BUT...before you execute KMeans...YOU MUST EXPLORE THE DATA!!!!!

You MUST visually explore the DISTRIBUTIONS NOT JUST by creating a pairs plot!!!

To see why...let's use a WIDE FORMAT plotting operation from Seaborn!

In [ ]:
sns.displot(data = penguins, aspect=2)

plt.show()

A boxplot from the WIDE FORMAT plotting operations makes it even easier to KNOW what's going on!

In [ ]:
sns.catplot(data = penguins, kind='box', aspect=2)

plt.show()

Why does it MATTER that 1 variable DOMINATES the magnitude and scale???

**REMEMBER** that KMeans considers SIMILAR to be based on DISTANCE!!!!

Distance depends on MAGNITUDE and Distance depends on SCALE!!!!

In [ ]:
penguins.describe()

If we use the columns with their RAW values, the magnitudes and scales will IMPACT the cluster results!!!!

We do NOT want the magnitudes and scales to impact things!!!

Essentially we want to work with variables that are ROUGHLY the same magnitude and scale!!!

We need to REMOVE the scale and thus TRANSFORM or MODIFY the variables!!!!!

That's why we must preprocess!!!!

## Preprocess

There are many different PREPROCESSING operations. But we will focus on 1 very important approach.

### Standardization

Standardizing means you are calculating the **Z-SCORE**.

What is the **Z-SCORE**? It's the NUMBER of standard deviations away from the average!!!!!!!

It requires 2 summary statistics:

* Sample average
* Sample standard deviation

The Z-score is calculated by:
* Centering - subtract the value from the SAMPLE AVERAGE
* Scale - divide the CENTERED value by the SAMPLE STANDARD DEVIATION



### 📏 The Math Behind Feature Scaling: Z-Score Standardization

Machine learning algorithms like K-Means and PCA calculate distances between data points. If one column is measured in "Age" (0 to 100) and another in "Salary" ($0 to $150,000), the massive salary numbers will completely overpower the age numbers! 

To fix this, we put all our columns on the exact same playing field using **Z-Score Standardization**. It centers our data around zero and scales it based on the standard deviation.

The mathematical formula for calculating a Z-Score is:

$$z_i = \frac{x_i - \mu}{\sigma}$$

**Let's break down what these symbols mean:**
* $z_i$ = The new standardized value (the z-score) for our specific data point.
* $x_i$ = The original, raw data point we are trying to scale.
* $\mu$ = The mean (average) of the entire column.
* $\sigma$ = The standard deviation of the entire column.

#### 🧠 What is this equation actually saying?
In plain English: *"Take a data point, subtract the average of the group from it (so the new center is exactly zero), and then divide it by the standard deviation."* The resulting Z-score literally just tells us: **"How many standard deviations is this point above or below the average?"** A Z-score of `0` means it's exactly average. A Z-score of `2.0` means it's unusually high, and `-1.5` means it's below average!

Demonstrate on one numeric column in `penguins`.

In [ ]:
penguins_z = penguins.copy()

In [ ]:
penguins_z.flipper_length_mm.mean()

In [ ]:
penguins_z.flipper_length_mm.std()

CENTER:

In [ ]:
( penguins_z.flipper_length_mm - penguins_z.flipper_length_mm.mean() )

SCALE:

In [ ]:
( penguins_z.flipper_length_mm - penguins_z.flipper_length_mm.mean() ) / penguins_z.flipper_length_mm.std()

Assign the z-score to a new column.

In [ ]:
penguins_z['flipper_length_zscore'] = ( penguins_z.flipper_length_mm - penguins_z.flipper_length_mm.mean() ) / penguins_z.flipper_length_mm.std()

Compare the original or RAW variable to its STANDARDIZED or Z-score value.

In [ ]:
penguins_z.loc[ :, ['flipper_length_mm', 'flipper_length_zscore'] ]

Compare the SUMMARY STATISTICS between the two.

In [ ]:
penguins_z.loc[ :, ['flipper_length_mm', 'flipper_length_zscore'] ].describe().round(3)

Standardization is also called NORMALIZATION...but I HATE that term because it sounds like we converting the variable to a NORMAL distribution!!

BUT THAT is COMPLETELY WRONG!!!! Standardizing does **NOT** change the distributional shape!!!

In [ ]:
sns.displot(data = penguins_z, x='flipper_length_mm', kind='hist', kde=True)

plt.show()

In [ ]:
sns.displot(data = penguins_z, x='flipper_length_zscore', kind='hist', kde=True)

plt.show()

Standardizing REMOVES the MAGNITUDE and SCALE and returns negative and positive values around 0.

## Preprocess with scikit-learn

scikit-learn has MANY functions to support machiine learning and data science.

scikit-learn has functions specializing in PREPROCESSING operations!

In [ ]:
from sklearn.preprocessing import StandardScaler

In [ ]:
%whos

Just like `KMeans()` all scikit-learn functions follow the following recipe:

* INITIALIZE the object based on assumptions
* FIT the OBJECT given a data set
* PREDICT or TRANSFORM a data set using the FITTED object

`KMeans()` returns cluster labels by PREDICTING, but `StandardScaler()` returns the STANDARDIZED columns by TRANSFORMING!!!!

Begin by INITIALIZING!!!

In [ ]:
pens_standardize = StandardScaler()

In [ ]:
type( pens_standardize )

We need to identify the columns that we will standardize.

In [ ]:
pens_features = penguins.select_dtypes('number').copy()

In [ ]:
pens_features.info()

Preprocessing CAN WORK with Pandas DataFrames!!!!!

FIT using the dataframe consisting of just the numeric columns!

In [ ]:
pens_standardize = pens_standardize.fit( pens_features )

TRANSFORM the numeric columns!

In [ ]:
Xpens = pens_standardize.transform( pens_features )

In [ ]:
type( Xpens )

In [ ]:
Xpens.shape

In [ ]:
pens_features.shape

You can accomplish INTIIALIZE, FIT, and TRANSFORM in a single line of code!

In [ ]:
StandardScaler().fit_transform( pens_features ).shape

Convert the returned NumPy array into a DataFrame to support visualizing with Seaborn.

In [ ]:
pd.DataFrame( Xpens, columns=pens_features.columns )

As a reminder...`body_mass_g` DOMINATES the magnitude and scale in the RAW data!

In [ ]:
sns.catplot(data = penguins, kind='box', aspect=2)

plt.show()

But does `body_mass_g` still dominate based on the STANDARDIZED or preprocessed variables?

In [ ]:
sns.catplot(data = pd.DataFrame(Xpens, columns=pens_features.columns), kind='box', aspect=2)

plt.show()

In [ ]:
sns.catplot(data = pd.DataFrame(Xpens, columns=pens_features.columns), kind='violin', aspect=2)

plt.show()

Standardizing does **NOT** modify the RELAIONSHIPS between the columns!

In [ ]:
sns.pairplot(data = pd.DataFrame(Xpens, columns=pens_features.columns))

plt.show()

The CORRELATION is UNCHANGED!!!!

In [ ]:
fig, ax = plt.subplots()

sns.heatmap( data = penguins.corr(numeric_only=True),
            vmin=-1, vmax=1, center=0,
            cmap='coolwarm',
            annot=True, annot_kws={'fontsize': 20},
            ax=ax)

plt.show()

In [ ]:
fig, ax = plt.subplots()

sns.heatmap( data = pd.DataFrame(Xpens, columns=pens_features.columns).corr(),
            vmin=-1, vmax=1, center=0,
            cmap='coolwarm',
            annot=True, annot_kws={'fontsize': 20},
            ax=ax)

plt.show()

In [ ]:
pd.DataFrame(Xpens, columns=pens_features.columns).corr()["bill_length_mm"]

In [ ]:
penguins.corr(numeric_only=True)["bill_length_mm"]